# Week 1 Analysis - Iris Dataset

Dataset source: https://www.kaggle.com/datasets/uciml/iris/data
Solo work by William Anthony (Bandung Institute of Technology, Indonesia).

Covers Day 1-6 tasks: data loading, EDA, feature prep, model training, evaluation, and pipeline.

In [ ]:
# William Anthony, from Indonesia, from Bandung Institute of Technology
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = Path("..") / "data" / "dataset.csv"
if not data_path.exists():
    data_path = Path.cwd() / "data" / "dataset.csv"

df = pd.read_csv(data_path)
df.head()

## Day 1 - Data Understanding

In [ ]:
df.shape
df.isna().sum()
df.duplicated().sum()
df.describe()

## Day 2 - Exploratory Data Analysis (EDA)

In [ ]:
ax = df["Species"].value_counts().plot(kind="bar", title="Class Distribution")
ax.set_xlabel("Species")
ax.set_ylabel("Count")
plt.show()

df.drop(columns=["Id"]).hist(figsize=(8, 6))
plt.tight_layout()
plt.show()

from pandas.plotting import scatter_matrix
scatter_matrix(df.drop(columns=["Id"]), figsize=(8, 8), diagonal="kde")
plt.show()

corr = df.drop(columns=["Id"]).corr()
plt.imshow(corr, cmap="coolwarm", interpolation="nearest")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title("Feature Correlation")
plt.tight_layout()
plt.show()

## Day 3 - Feature Preparation

In [ ]:
features = df.drop(columns=["Species"])
if "Id" in features.columns:
    features = features.drop(columns=["Id"])

X = features
y = df["Species"]

X.head()

## Day 4 - Model Training

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "log_reg": Pipeline(
        [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
    ),
    "svm": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", gamma="scale"))]),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42),
}

scores = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    scores[name] = {"mean": cv_scores.mean(), "std": cv_scores.std()}

pd.DataFrame(scores).T.sort_values("mean", ascending=False)

## Day 5 - Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

best_name = max(scores, key=lambda k: scores[k]["mean"])
best_model = models[best_name]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
 )
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

accuracy_score(y_test, y_pred)
confusion_matrix(y_test, y_pred)
print(classification_report(y_test, y_pred))

## Day 6 - Pipeline and Model Export

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

if isinstance(best_model, Pipeline):
    final_model = clone(best_model)
else:
    final_model = Pipeline([("scaler", StandardScaler()), ("model", clone(best_model))])

final_model.fit(X, y)

model_path = Path("..") / "models" / "iris_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, model_path)

model_path